# Revelstoke resort capacity and workforce housing

This report's prose is assembled in code cells from data/processed files, with every factual sentence carrying a citation from `cite()`. Per CLAUDE.md rule 7, no number below is typed by hand: everything is an f-string built from a variable loaded from data/processed. `tools/check_citations.py --report` enforces that every claim ID cited here is human-verified with a source that has passed verification.

In [ ]:
research_dir = "research"
processed_dir = "data/processed"

## Load the ledger

cite() will raise if any claim used below is not human-verified with a passed source, which is the guarantee this report rests on.

In [ ]:
import sys

sys.path.insert(0, "src")
from resort.ledger import cite, read_ledger

ledger = read_ledger(
    claims_path=f"{research_dir}/claims.csv",
    sources_path=f"{research_dir}/sources.csv",
)
print(f"{len(ledger['claims'])} claims loaded")

## A small citation helper

Several sentences below cite multiple claims that share the same source, or sources with the same organizational author and no year, which `cite()` alone would print as repeated identical labels back to back (e.g. two copies of `(City of Revelstoke)`). `cite_many` calls `cite()` for every claim ID (so every one is still individually checked as human-verified with a passed source) but only prints each distinct label once, in first-seen order.

In [ ]:
def cite_many(claim_ids, ledger):
    labels = [cite(claim_id, ledger) for claim_id in claim_ids]
    seen = []
    for label in labels:
        if label not in seen:
            seen.append(label)
    return " ".join(seen)

## Load every data/processed output this report draws on

One place to load everything, so each section below only formats numbers it already has, rather than reading files throughout the report.

In [ ]:
import json

import pandas as pd

ccc_by_phase = pd.read_csv(f"{processed_dir}/02_ccc_by_phase.csv").set_index("phase")
lift_reproduction = pd.read_csv(f"{processed_dir}/03_lift_ccc_reproduction.csv")
phase_reproduction = pd.read_csv(f"{processed_dir}/03_ccc_phase_reproduction_summary.csv").set_index("phase")
elevation_bands = pd.read_csv(f"{processed_dir}/04_elevation_bands.csv")
with open(f"{processed_dir}/04_historical_snowfall.json", encoding="utf-8") as f:
    historical_snowfall = json.load(f)
employee_estimates = pd.read_csv(f"{processed_dir}/05_employee_estimates.csv").set_index("phase")
with open(f"{processed_dir}/02_employee_housing_phase2.json", encoding="utf-8") as f:
    employee_housing = json.load(f)
housing_need = pd.read_csv(f"{processed_dir}/02_housing_need_by_component.csv")
with open(f"{processed_dir}/06_rental_vacancy.json", encoding="utf-8") as f:
    rental_vacancy = json.load(f)
zoning = pd.read_csv(f"{processed_dir}/07_zoning_by_parcel_count.csv")
with open(f"{processed_dir}/07_resort_lands_dpa.json", encoding="utf-8") as f:
    resort_lands_dpa = json.load(f)
scenario = pd.read_csv(f"{processed_dir}/08_scenario_table.csv").set_index("phase")
print("all inputs loaded")

## Scope

Displayed via `Markdown()` from a code cell rather than typed directly, even though this section has no numbers, to keep the whole report in one consistent pattern.

In [ ]:
from IPython.display import Markdown, display

display(Markdown(f'''
Revelstoke Mountain Resort (RMR) publishes a master plan governing its
on-mountain development. This report follows one chain from that plan
through to a workforce-housing estimate: terrain and lifts set skier
capacity, snow reliability would adjust that capacity, capacity sets
workforce size, workforce sets housing demand, and land capacity shows
where that housing could go. Two links in that chain are incomplete as
noted in their own sections below; this report states that plainly
rather than filling the gap with an unsourced number.
'''))

## Terrain and capacity

States the appendix's formula and totals, then the actual reproduction result computed in step 03, distinguishing the cited claim from the notebook's own check.

In [ ]:
buildout_ccc = int(ccc_by_phase.loc["Buildout", "ccc_skiers"])
n_lift_rows = len(lift_reproduction)
max_lift_diff = (lift_reproduction["computed_ccc"] - lift_reproduction["stated_ccc"]).abs().max()
max_phase_diff = (phase_reproduction["computed_ccc_skiers"] - phase_reproduction["ccc_skiers"]).abs().max()

display(Markdown(f'''
RMR's master plan appendix states a comfortable carrying capacity (CCC)
of {buildout_ccc:,} skiers at full buildout {cite("C025", ledger)}, using a
formula that compares each lift's vertical transport capacity against
skier demand for vertical, weighted by ability level {cite("C007", ledger)}
{cite("C024", ledger)}. This report reproduced every one of the appendix's
{n_lift_rows} (phase, lift) rows from that formula: the largest per-lift
difference was {max_lift_diff:.1f} skiers and the largest phase-total
difference was {max_phase_diff:.0f} skiers, both within rounding.

The resort's own terrain and elevation figures
{cite_many(["C032", "C033"], ledger)} give the elevation range used for
snow reliability below.
'''))

## Snow reliability

Reports what was actually built (elevation bands, a historical baseline) and is explicit that the projection half of this question is not modeled, rather than silently omitting it.

In [ ]:
rmr_low = historical_snowfall["rmr_annual_m_low"]
rmr_high = historical_snowfall["rmr_annual_m_high"]
n_bands = len(elevation_bands)

display(Markdown(f'''
RMR's own historical snowfall figures are {rmr_low}-{rmr_high} metres a
year {cite("C034", ledger)}. This report built {n_bands} elevation bands
between the master plan's own stated elevation points (from
{cite("C033", ledger)}, already cited above) as the basis for a
snow-reliability indicator.

A dataset exists for projecting snowfall under future emissions
scenarios {cite("C005", ledger)} {cite("C006", ledger)}, but this report
does not use it: the specific derived snow-water-equivalent product's
listed access path did not resolve this session, and the base dataset
alone does not include snow variables {cite("C035", ledger)}
{cite("C036", ledger)}. No projected or snow-adjusted capacity figure is
stated anywhere in this report as a result.
'''))

## Workforce

Cites the one comparator ratio this report's employee estimate rests on, and states the sensitivity band's own nature (a modeling choice, not a second source) directly in the same sentence as its citation.

In [ ]:
low = int(employee_estimates.loc["Buildout", "employees_low"])
mid = int(employee_estimates.loc["Buildout", "employees_mid"])
high = int(employee_estimates.loc["Buildout", "employees_high"])
beds_low = employee_housing["total_beds_min"]
beds_high = employee_housing["total_beds_max"]
need_min = int(employee_estimates.loc["Buildout", "needing_housing_mid_min_beds"])
need_max = int(employee_estimates.loc["Buildout", "needing_housing_mid_max_beds"])

# Quote each headcount/housing claim's own wording directly from the
# ledger rather than retyping its figures by hand, so the report cannot
# silently drift from what the claim actually says.
c010_text = ledger["claims"]["C010"]["claim"].rstrip(".")
c009_text = ledger["claims"]["C009"]["claim"].rstrip(".")
c011_text = ledger["claims"]["C011"]["claim"].rstrip(".")
c012_text = ledger["claims"]["C012"]["claim"].rstrip(".")

display(Markdown(f'''
RMR's stated employee figures range from "{c010_text}" {cite("C010", ledger)}
to "{c009_text}" {cite("C009", ledger)}; this report carries both rather
than picking one, since neither source rules out the other describing a
different subset or point in time.

To estimate workforce at buildout capacity, this report uses the one
public source found giving both a resort's approved capacity and its
peak employee count together: Parks Canada's Sunshine Village site
guidelines {cite("C040", ledger)}. Applying that single ratio with a
symmetric sensitivity band (a modeling choice, not a second source)
gives {low:,}-{high:,} employees at buildout ({mid:,} at the band's
centre).

The master plan's own Phase 2 employee housing plan provides
{beds_low:,}-{beds_high:,} beds {cite("C031", ledger)}. RMR itself has
separately stated "{c011_text}" {cite("C011", ledger)} and, in an earlier
2022 plan, "{c012_text}" {cite("C012", ledger)}. Subtracting the housing
plan's own range from the buildout employee estimate leaves roughly
{need_min:,}-{need_max:,} workers needing housing elsewhere, at the
sensitivity band's centre.
'''))

## Housing supply and affordability

Reports the City's own housing-need and rental-vacancy figures without projecting them forward.

In [ ]:
need_5yr = int(housing_need["years_5"].sum())
need_20yr = int(housing_need["years_20"].sum())
vacancy = rental_vacancy["vacancy_rate_2021_pct"]
healthy_low = rental_vacancy["healthy_vacancy_low_pct"]
healthy_high = rental_vacancy["healthy_vacancy_high_pct"]

display(Markdown(f'''
The City of Revelstoke's own Housing Needs Report states a 5-year
housing need of {need_5yr:,} units and a 20-year need of {need_20yr:,}
units {cite_many(["C003", "C026", "C027", "C028", "C029"], ledger)}, against
a 2021 rental vacancy rate of {vacancy}%, below the
{healthy_low}-{healthy_high}% range the report itself calls healthy.

The resort and the municipality are both named exempt from the
province's short-term-rental principal-residence requirement, while the
surrounding rural electoral area is not {cite("C013", ledger)}
{cite("C014", ledger)}. This report does not compare rents against
resort wages: no resort-specific wage figure or dollar-value renter
affordability table was found this session.
'''))

## Land capacity

Reports the real zoning distribution and resort-lands area, and states plainly that this does not amount to a unit-capacity figure.

In [ ]:
n_zones = len(zoning)
top_zone = zoning.sort_values("parcel_count", ascending=False).iloc[0]
dpa_ha = resort_lands_dpa["area_hectares"]

display(Markdown(f'''
Querying the City of Revelstoke's own zoning data live shows
{n_zones} distinct zone codes across the municipality, the largest by
parcel count being {top_zone["zoning_name"]} ({int(top_zone["parcel_count"]):,}
parcels) {cite("C049", ledger)}. The Official Community Plan's Resort
Lands Development Permit Area covers about {dpa_ha:,.0f} hectares
{cite_many(["C046", "C050"], ledger)}, live data made possible by
the City's own Zoning and Parcel Fabric map services
{cite_many(["C045", "C047"], ledger)}.

This report does not convert either figure into a number of housing
units: the zoning bylaw's density provisions and a way to identify
which specific parcels are vacant or underused were not found this
session. The Oscar Lands Master Plan, approved in February 2024
{cite("C015", ledger)}, has not been read as a document to check whether
it states its own unit-capacity figures.
'''))

## What this report does not claim

A plain list of the open gaps, so a reader does not have to infer them from what is missing.

In [ ]:
display(Markdown(f'''
- No snow-adjusted or climate-projected capacity figure exists for any
  phase.
- No land-capacity figure in units of housing exists.
- The workforce estimate rests on one comparator resort's ratio, not
  RMR's own, and on a sensitivity band that is a modeling choice.
- RMR's own current headcount is carried as two disagreeing figures
  {cite("C009", ledger)} {cite("C010", ledger)}, not resolved to one.
- This report's own verified demand-buffer figure for Revelstoke is
  1.63 {cite("C028", ledger)}. A secondary source (not itself
  independently verified this session) states a different, unconfirmed
  figure for what it calls the same provincial method; that
  disagreement is recorded in steps/06's Open issues and is not
  resolved here.
- Ownership affordability figures from the Housing Needs Report are not
  yet human-verified and are not cited here.
'''))

## Write outputs

No new data/processed file: this report only reads existing outputs and cites existing claims, so there is nothing new to write except confirmation of which claims it used.

In [ ]:
import os

cited_claim_ids = [
    "C025", "C007", "C024", "C032", "C033", "C034", "C005", "C006", "C035", "C036",
    "C010", "C009", "C040", "C031", "C011", "C012", "C003", "C026", "C027", "C028",
    "C029", "C013", "C014", "C049", "C046", "C050", "C045", "C047", "C015",
]

os.makedirs(processed_dir, exist_ok=True)
with open(f"{processed_dir}/09_cited_claims.json", "w", encoding="utf-8") as f:
    json.dump(sorted(set(cited_claim_ids), key=lambda c: int(c[1:])), f, indent=2)
print(f"wrote 09_cited_claims.json ({len(set(cited_claim_ids))} distinct claims)")

## Checks

Every claim ID this report cites must actually be human-verified with a passed source (cite() already enforces this per call, so if the report rendered above without error, this cell mainly documents the guarantee explicitly).

In [ ]:
for claim_id in cited_claim_ids:
    cite(claim_id, ledger)  # raises LedgerError if not human-verified + passed
print(f"checks passed: all {len(set(cited_claim_ids))} cited claims are human-verified with a passed source")

## Versions

In [ ]:
import importlib.metadata
import sys

print("python", sys.version)
for pkg in ["pandas"]:
    print(pkg, importlib.metadata.version(pkg))